# 04 · Gold

Silver → business tables in `data/gold/`, fully recomputed on each run:

| Table | Grain |
|---|---|
| `enem_fact_candidates` | one row per candidate, with average score and performance tier |
| `enem_performance_by_state` | year × state |
| `enem_performance_by_school` | year × state × school type × gender |

**Business rule:** `average_score` only exists for candidates who attended all four objective exams.
Absent candidates get the `Absent` tier and stay out of the averages instead of counting as zero.

In [ ]:
%run ./00_setup.ipynb

In [ ]:
from pyspark.sql.functions import avg, col, count, round as spark_round, stddev, when

EXAM_SCORES = ["score_natural_sciences", "score_humanities", "score_languages", "score_math", "score_essay"]

silver = spark.read.format("delta").load(SILVER_TABLE)

### Fact table: candidates

In [ ]:
present_all_exams = (
    (col("presence_natural_sciences") == 1)
    & (col("presence_humanities") == 1)
    & (col("presence_languages") == 1)
    & (col("presence_math") == 1)
)
average = sum(col(c) for c in EXAM_SCORES) / len(EXAM_SCORES)

fact_candidates = (
    silver.withColumn("is_present_all_exams", present_all_exams)
    .withColumn("average_score", when(col("is_present_all_exams"), spark_round(average, 2)))
    .withColumn(
        "performance_tier",
        when(col("average_score").isNull(), "Absent")
        .when(col("average_score") >= 750, "Excellent")
        .when(col("average_score") >= 600, "Good")
        .when(col("average_score") >= 450, "Average")
        .otherwise("Below Average"),
    )
)

### Aggregates: by state and by school type

In [ ]:
# avg() ignores nulls, so absent candidates don't pull the averages down
score_averages = [spark_round(avg(c), 2).alias(c.replace("score_", "avg_")) for c in EXAM_SCORES]

performance_by_state = fact_candidates.groupBy("year", "exam_state").agg(
    count("*").alias("total_enrolled"),
    count(when(col("is_present_all_exams"), True)).alias("total_present_all_exams"),
    *score_averages,
    spark_round(stddev("score_essay"), 2).alias("stddev_essay"),
    spark_round(avg("average_score"), 2).alias("overall_avg_score"),
)

school_type = (
    when(col("school_type") == 1, "Not Specified")
    .when(col("school_type") == 2, "Public")
    .when(col("school_type") == 3, "Private")
    .otherwise("Other")
)

performance_by_school = (
    fact_candidates.withColumn("school_type_description", school_type)
    .groupBy("year", "exam_state", "school_type_description", "gender")
    .agg(
        count("*").alias("total_candidates"),
        *score_averages,
        spark_round(avg("average_score"), 2).alias("overall_avg_score"),
    )
)

Data quality check: absent candidates must not have an average score.

In [ ]:
assert fact_candidates.filter(~col("is_present_all_exams") & col("average_score").isNotNull()).isEmpty()
print("Quality check passed")

In [ ]:
gold_tables = {
    # name: (DataFrame, partition columns) - only the candidate-level fact table is large enough to partition
    "enem_fact_candidates": (fact_candidates, ["year", "exam_state"]),
    "enem_performance_by_state": (performance_by_state, []),
    "enem_performance_by_school": (performance_by_school, []),
}

for name, (df, partitions) in gold_tables.items():
    (
        df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")  # Gold is recomputed: allow schema/partition changes
        .partitionBy(*partitions)
        .save(str(GOLD_DIR / name))
    )
    print(f"Saved {name}")

### Results

In [ ]:
gold = {name: spark.read.format("delta").load(str(GOLD_DIR / name)) for name in gold_tables}

gold["enem_performance_by_state"].orderBy(col("overall_avg_score").desc()).show(truncate=False)

In [ ]:
gold["enem_performance_by_school"].show(10, truncate=False)

In [ ]:
gold["enem_fact_candidates"].select(
    "registration_id", "exam_state", "is_present_all_exams", "average_score", "performance_tier"
).show(10)

Any table can be queried with SQL through a temporary view.

In [ ]:
gold["enem_fact_candidates"].createOrReplaceTempView("fact_candidates")

spark.sql("""
    SELECT performance_tier, COUNT(*) AS candidates, ROUND(AVG(average_score), 2) AS avg_score
    FROM fact_candidates
    GROUP BY performance_tier
    ORDER BY avg_score DESC NULLS LAST
""").show()

In [ ]:
spark.stop()